# NumCompute quickstart

End-to-end demo of the `numcompute` package.

In [1]:
%pip install -e ..

Obtaining file:///Users/sandunigalagoda/Documents/Uni/1st%20Semester/PAI/Code_Cube-COMP-5004-Assignment-2.1-Programming-Task-1
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for numcompute (pyproject.toml) ... done
  Created wheel for numcompute: filename=numcompute-0.1.0-0.editable-py3-none-any.whl size=3831 sha256=49cd6db81c943d835775c7df999521effa07f451f689edfd72800d56082b9866
  Stored in directory: /private/var/folders/9w/kqvbhpfx7bj2cjvzh7vd7mcm0000gn/T/pip-ephem-wheel-cache-bzj3yvfu/wheels/97/63/6f/232200944c621a3f84234136c5e92288cc8d50ba6bf4085b4c
Successfully built numcompute
  Attempting uninstall: numcompute
    Found existing installation: numcompute 0.1.0
    Uninstalling numcompute-0.1.0:
      Successfully uninstalled numcompute-0.1.0
Note: you may need to restart the kernel to use updated pack

In [2]:
import numpy as np
from IPython.display import Markdown, display
from numcompute import io as nc_io
from numcompute import sort_search, rank, stats, utils, benchmarking
from numcompute.preprocessing import (
    StandardScaler, SimpleImputer, OneHotEncoder)
from numcompute.optim import grad, jacobian

## Load, inspect, clean and prepare the data

In [3]:
path = "data/penguins.csv"
names = nc_io.read_header(path)
for i, name in enumerate(names):
    print(i, name)

0 species
1 island
2 bill_length_mm
3 bill_depth_mm
4 flipper_length_mm
5 body_mass_g
6 sex


### Load the file
`load_mixed_csv` reads every value as text first, then converts. It returns
two arrays: 
- `X_num` : (numbers, with NaN where a value was missing) 
- `X_cat` : (text categories).

In [4]:
num_cols = [2, 3, 4, 5]    
cat_cols = [0, 1, 6] 

X_num, X_cat = nc_io.load_mixed_csv(path, num_cols, cat_cols, skip_header=1)

num_names = [names[i] for i in num_cols]
cat_names = [names[i] for i in cat_cols]

print("numeric    :", num_names, X_num.shape, X_num.dtype)
print("categorical:", cat_names, X_cat.shape, X_cat.dtype)

numeric    : ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'] (344, 4) float64
categorical: ['species', 'island', 'sex'] (344, 3) <U9


In [5]:
print(f"{'':>3}" + "".join(f"{n[:15]:>17}" for n in num_names)
      + "".join(f"{n:>12}" for n in cat_names))
for i in range(5):
    print(f"{i:>3}" + "".join(f"{v:>17}" for v in X_num[i])
          + "".join(f"{v:>12}" for v in X_cat[i]))

      bill_length_mm    bill_depth_mm  flipper_length_      body_mass_g     species      island         sex
  0             39.1             18.7            181.0           3750.0      Adelie   Torgersen        MALE
  1             39.5             17.4            186.0           3800.0      Adelie   Torgersen      FEMALE
  2             40.3             18.0            195.0           3250.0      Adelie   Torgersen      FEMALE
  3              nan              nan              nan              nan      Adelie   Torgersen            
  4             36.7             19.3            193.0           3450.0      Adelie   Torgersen      FEMALE


### Missing data
Missing numerical data must be filled before scaling.

For missing categorical data, such as `sex`, the value is left
blank in the file. The next cell renames it to `unknown` so it reads clearly. It
stays a category of its own, so no penguin is dropped.

In [6]:
missing = np.isnan(X_num)
print("missing per column:", dict(zip(num_names, missing.sum(axis=0).tolist())))
rows_with_gaps = np.flatnonzero(missing.any(axis=1))
print("rows with a missing number:", rows_with_gaps.size, "of", X_num.shape[0])
print("row numbers:", rows_with_gaps.tolist())

missing per column: {'bill_length_mm': 2, 'bill_depth_mm': 2, 'flipper_length_mm': 2, 'body_mass_g': 2}
rows with a missing number: 2 of 344
row numbers: [3, 339]


### Rename blank categories
A blank category becomes `unknown`. The counts show how many penguins fall
into each category.

In [7]:
X_cat = np.where(X_cat == "", "unknown", X_cat)

for j, name in enumerate(cat_names):
    values, counts = np.unique(X_cat[:, j], return_counts=True)
    print(name, dict(zip(values.tolist(), counts.tolist())))


species {'Adelie': 152, 'Chinstrap': 68, 'Gentoo': 124}
island {'Biscoe': 168, 'Dream': 124, 'Torgersen': 52}
sex {'FEMALE': 165, 'MALE': 168, 'unknown': 11}


### Fill missing numerical data
`SimpleImputer` replaces each missing number
with the median of its column. The median is a good choice because
extreme values do not affect it much.

In [8]:
first_gap = int(rows_with_gaps[0])
imputer = SimpleImputer("median").fit(X_num)
filled = imputer.transform(X_num)

print("missing numbers left:", int(np.isnan(filled).sum()))
print("medians used:", dict(zip(num_names, imputer.statistics_.round(1).tolist())))
print(f"row {first_gap} before:", X_num[first_gap])
print(f"row {first_gap} after :", filled[first_gap])

missing numbers left: 0
medians used: {'bill_length_mm': 44.4, 'bill_depth_mm': 17.3, 'flipper_length_mm': 197.0, 'body_mass_g': 4050.0}
row 3 before: [nan nan nan nan]
row 3 after : [  44.45   17.3   197.   4050.  ]


### Scale the numbers
`StandardScaler` rescales each column so it has
mean 0 and standard deviation 1.

In [9]:
scaler = StandardScaler().fit(filled)
Z = scaler.transform(filled)

print("mean per column:", Z.mean(axis=0).round(3))
print("std per column :", Z.std(axis=0).round(3))
print("row 0 raw   :", filled[0])
print("row 0 scaled:", Z[0].round(2))
print("undo scaling works:", np.allclose(scaler.inverse_transform(Z), filled))

mean per column: [-0. -0.  0.  0.]
std per column : [1. 1. 1. 1.]
row 0 raw   : [  39.1   18.7  181.  3750. ]
row 0 scaled: [-0.89  0.79 -1.42 -0.56]
undo scaling works: True


### Encode the categories
`OneHotEncoder` gives every category its own 0/1 column.

In [10]:
enc = OneHotEncoder().fit(X_cat)
H = enc.transform(X_cat)

for name, cats in zip(cat_names, enc.categories_):
    print(name, "->", cats.tolist())
print("encoded shape:", H.shape)
print("row 0:", X_cat[0].tolist(), "->", H[0])
print("every row has one 1 per column:",
      bool(np.all(H.sum(axis=1) == len(cat_cols))))

species -> ['Adelie', 'Chinstrap', 'Gentoo']
island -> ['Biscoe', 'Dream', 'Torgersen']
sex -> ['FEMALE', 'MALE', 'unknown']
encoded shape: (344, 9)
row 0: ['Adelie', 'Torgersen', 'MALE'] -> [1. 0. 0. 0. 0. 1. 0. 1. 0.]
every row has one 1 per column: True


### Combine
The scaled numbers and the encoded categories are joined side by side into
one feature matrix.

In [11]:
features = np.hstack([Z, H])
print("feature matrix:", features.shape)

feature matrix: (344, 13)


## Sort, search and rank

This part uses `sort_search` (sorting, top-k, quickselect, binary search) and
`rank` (ranking with ties, percentiles). These functions reject NaN instead of
guessing where it belongs, so this part uses the penguins with all four
measurements.

In [12]:
np.set_printoptions(precision=2, suppress=True)

complete = ~np.isnan(X_num).any(axis=1)
clean = X_num[complete]
species = X_cat[complete, 0]
bill, flipper, mass = clean[:, 0], clean[:, 2], clean[:, 3]

print("penguins with every measurement:", clean.shape[0], "of", X_num.shape[0])

penguins with every measurement: 342 of 344


### Top-k
`topk` to find the k largest (or smallest) values in
O(n) without sorting the whole array; only those k are then sorted. It also
returns their positions, so they can be matched to the other columns.

In [13]:
heaviest, idx = sort_search.topk(mass, 5)
print("5 heaviest (g):", heaviest)
print("species       :", species[idx])

lightest, idx = sort_search.topk(mass, 3, largest=False)
print("3 lightest (g):", lightest)
print("species       :", species[idx])

5 heaviest (g): [6300. 6050. 6000. 6000. 5950.]
species       : ['Gentoo' 'Gentoo' 'Gentoo' 'Gentoo' 'Gentoo']
3 lightest (g): [2700. 2850. 2850.]
species       : ['Chinstrap' 'Adelie' 'Adelie']


### Quickselect
`quickselect(values, k)` returns the k-th smallest value (`k=0` is the
minimum) by partitioning around random pivots. That takes O(n) on average,
instead of O(n log n) for a full sort. The answer is checked against sorting.

In [14]:
n = mass.size
k = n // 2
middle = sort_search.quickselect(mass, k, seed=0)

print(f"k = {k} of {n}: {middle} g")
print("same as sorting:", middle == sort_search.stable_sort(mass)[k])
print("k = 0 (lightest):", sort_search.quickselect(mass, 0, seed=0))
print(f"k = {n - 1} (heaviest):", sort_search.quickselect(mass, n - 1, seed=0))

k = 171 of 342: 4050.0 g
same as sorting: True
k = 0 (lightest): 2700.0
k = 341 (heaviest): 6300.0


### Multi-key sort
Rows sorted by flipper length, with body mass breaking ties. The last three
rows all have 178 mm flippers, so mass decides their order. Rows that are equal
on both keys keep their order from the file (the sort is stable).

In [15]:
by_flipper_then_mass = sort_search.multi_key_sort(clean, [2, 3])

print("".join(f"{n[:15]:>17}" for n in num_names))
for row in by_flipper_then_mass[:6]:
    print("".join(f"{v:>17}" for v in row))

   bill_length_mm    bill_depth_mm  flipper_length_      body_mass_g
             37.9             18.6            172.0           3150.0
             37.8             18.3            174.0           3400.0
             40.2             17.0            176.0           3450.0
             33.1             16.1            178.0           2900.0
             39.5             16.7            178.0           3250.0
             46.1             18.2            178.0           3250.0


### Binary search
`binary_search` takes a sorted array and returns, for each query, the leftmost
position where it fits and whether it is present. Here that position is also
the number of penguins with a shorter flipper. All queries are searched in one
call.

In [16]:
flipper_sorted = sort_search.stable_sort(flipper)
queries = np.array([172, 190, 204, 231, 240])
index, exists = sort_search.binary_search(flipper_sorted, queries)

print(f"{'query (mm)':>10}{'position':>10}{'present':>9}")
for q, i, e in zip(queries, np.asarray(index), np.asarray(exists)):
    print(f"{q:>10}{i:>10}{str(e):>9}")

query (mm)  position  present
       172         0     True
       190        77     True
       204       209    False
       231       341     True
       240       342    False


### Ranking with ties
Ranks start at 1 for the lightest penguin. Body mass is recorded in steps of
25 g, so many penguins share a mass. `method` decides what tied values get:
- `average`: the mean of the positions they cover
- `dense`: the same rank, with no ranks skipped after them
- `ordinal`: a different rank each, in the order they appear in the file

In [17]:
print("different masses:", np.unique(mass).size, "among", mass.size, "penguins")

at_3800 = mass == 3800
print("penguins weighing 3800 g:", int(at_3800.sum()))
for method in ["average", "dense", "ordinal"]:
    print(f"{method:<8}", rank.rank(mass, method)[at_3800])

different masses: 94 among 342 penguins
penguins weighing 3800 g: 12
average  [135.5 135.5 135.5 135.5 135.5 135.5 135.5 135.5 135.5 135.5 135.5 135.5]
dense    [34 34 34 34 34 34 34 34 34 34 34 34]
ordinal  [130 131 132 133 134 135 136 137 138 139 140 141]


### Percentiles
`percentile(data, q)` takes `q` from 0 to 100. When a percentile falls between
two data points, `interpolation` picks the answer: `linear` blends the two,
`lower` and `higher` take one of them, and `midpoint` takes their average.

In [18]:
qs = [25, 50, 75]

print(f"{'':<14}" + "".join(f"{q:>8}" for q in qs))
for method in ["linear", "lower", "higher", "midpoint"]:
    values = np.asarray(rank.percentile(bill, qs, method))
    print(f"{method:<14}" + "".join(f"{v:>8.2f}" for v in values))
print(f"{'np.percentile':<14}" + "".join(f"{v:>8.2f}" for v in np.percentile(bill, qs)))

                    25      50      75
linear           39.23   44.45   48.50
lower            39.20   44.40   48.50
higher           39.30   44.50   48.50
midpoint         39.25   44.45   48.50
np.percentile    39.23   44.45   48.50


## Statistics

`stats` functions work on the whole array or along an axis:
- `axis=None`: one number
- `axis=0`: one value per column, shape `(n_features,)`
- `axis=1`: one value per row, shape `(n_samples,)`

`nan_policy` decides what happens with NaN: `propagate` (the result is NaN),
`omit` (NaN is ignored) or `raise` (an error). This part uses the raw `X_num`,
including the rows with missing values.

In [19]:
print("propagate:", stats.mean(X_num, axis=0))
print("omit     :", stats.mean(X_num, axis=0, nan_policy="omit"))
try:
    stats.mean(X_num, axis=0, nan_policy="raise")
except ValueError as err:
    print("raise    : ValueError:", err)

print()
print("X_num shape:", X_num.shape)
print("axis=None  :", np.shape(stats.mean(X_num, nan_policy="omit")))
print("axis=0     :", stats.mean(X_num, axis=0, nan_policy="omit").shape)
print("axis=1     :", stats.mean(X_num, axis=1, nan_policy="omit").shape)

propagate: [nan nan nan nan]
omit     : [  43.92   17.15  200.92 4201.75]
raise    : ValueError: Input contains NaN and nan_policy='raise'

X_num shape: (344, 4)
axis=None  : ()
axis=0     : (4,)
axis=1     : (344,)


### Summary of each measurement
`std` uses `ddof=1` here, the sample standard deviation.

In [20]:
summary = {
    "mean": stats.mean(X_num, axis=0, nan_policy="omit"),
    "median": stats.median(X_num, axis=0, nan_policy="omit"),
    "variance": stats.var(X_num, axis=0, nan_policy="omit"),
    "std (ddof=1)": stats.std(X_num, axis=0, ddof=1, nan_policy="omit"),
    "min": stats.min(X_num, axis=0, nan_policy="omit"),
    "max": stats.max(X_num, axis=0, nan_policy="omit"),
}
print(f"{'':<14}" + "".join(f"{n[:15]:>17}" for n in num_names))
for label, row in summary.items():
    print(f"{label:<14}" + "".join(f"{v:>17.2f}" for v in row))

                 bill_length_mm    bill_depth_mm  flipper_length_      body_mass_g
mean                      43.92            17.15           200.92          4201.75
median                    44.45            17.30           197.00          4050.00
variance                  29.72             3.89           197.15        641250.58
std (ddof=1)               5.46             1.97            14.06           801.95
min                       32.10            13.10           172.00          2700.00
max                       59.60            21.50           231.00          6300.00


In [21]:
for s in np.unique(species):
    m = mass[species == s]
    print(f"{s:<10} n={m.size:>3}   mean mass {stats.mean(m):7.1f} g"
          f"   std {stats.std(m, ddof=1):6.1f} g")

Adelie     n=151   mean mass  3700.7 g   std  458.6 g
Chinstrap  n= 68   mean mass  3733.1 g   std  384.3 g
Gentoo     n=123   mean mass  5076.0 g   std  504.1 g


### Histogram
`histogram` returns the count in each bin and the bin edges (one more edge
than bins). NaN is dropped by default, since it belongs in no bin.

In [22]:
flipper_raw = X_num[:, 2]
counts, edges = stats.histogram(flipper_raw, bins=8)

print("counted:", counts.sum(), "| NaN dropped:", int(np.isnan(flipper_raw).sum()))
for c, lo, hi in zip(counts, edges[:-1], edges[1:]):
    print(f"{lo:6.1f} to {hi:6.1f} mm | {'#' * int(c // 2):<46} {c}")

counted: 342 | NaN dropped: 2
 172.0 to  179.4 mm | ####                                           8
 179.4 to  186.8 mm | ####################                           40
 186.8 to  194.1 mm | #############################################  91
 194.1 to  201.5 mm | ##############################                 61
 201.5 to  208.9 mm | ###########                                    23
 208.9 to  216.2 mm | ##############################                 60
 216.2 to  223.6 mm | ##################                             37
 223.6 to  231.0 mm | ###########                                    22


### Quantiles
`quantile` takes `q` from 0 to 1 (`rank.percentile` uses 0 to 100) and works
along an axis. With three values of `q` and `axis=0` the result is
`(3, n_features)`: one row per `q`.

In [23]:
Q = stats.quantile(X_num, [0.25, 0.5, 0.75], axis=0, nan_policy="omit")

print("shape:", Q.shape)
print(f"{'':<6}" + "".join(f"{n[:15]:>17}" for n in num_names))
for label, row in zip(["25%", "50%", "75%"], Q):
    print(f"{label:<6}" + "".join(f"{v:>17.2f}" for v in row))
print("bill length matches rank.percentile:",
      np.allclose(Q[:, 0], rank.percentile(bill, [25, 50, 75])))

shape: (3, 4)
         bill_length_mm    bill_depth_mm  flipper_length_      body_mass_g
25%               39.23            15.60           190.00          3550.00
50%               44.45            17.30           197.00          4050.00
75%               48.50            18.70           213.00          4750.00
bill length matches rank.percentile: True


### Streaming statistics
`RunningStats` updates the mean and variance one chunk at a time (Welford's
method), so the whole dataset never has to be in memory. Here the data is fed
in batches of 50 rows with `utils.batch_iter`, and the result is compared with
the full-array `stats` functions.

In [24]:
rs = stats.RunningStats()
for batch in utils.batch_iter(clean, 50):
    rs.update(batch)

print("samples seen:", rs.count)
print("mean        :", rs.mean)
print("variance    :", rs.variance())
print("same as stats.mean:", np.allclose(rs.mean, stats.mean(clean, axis=0)))
print("same as stats.var :", np.allclose(rs.variance(), stats.var(clean, axis=0)))

samples seen: 342
mean        : [  43.92   17.15  200.92 4201.75]
variance    : [    29.72      3.89    197.15 641250.58]
same as stats.mean: True
same as stats.var : True


## Gradients and Jacobians
`grad(f, x)` estimates the gradient of a scalar function by finite differences
(central by default, step `h=1e-5`). `jacobian(F, x)` does the same for a
function with several outputs and returns shape `(outputs, inputs)`. The scalar
function is the mean squared error of a straight-line model that predicts body
mass from the other three scaled measurements, so the exact gradient is known
and can be compared.

In [25]:
A = Z[:, :3]
t = Z[:, 3]
loss = lambda w: np.mean((t - A @ w) ** 2)

w0 = np.zeros(3)
g = grad(loss, w0)
g_exact = -2 / len(t) * A.T @ t
print("scalar function")
print("numeric:", g)
print("exact  :", g_exact)
print("match  :", np.allclose(g, g_exact, atol=1e-6))

J = jacobian(lambda w: A[:5] @ w, w0)
print("\nvector function (5 outputs, 3 inputs)")
print("jacobian shape:", J.shape)
print("equals first 5 rows of A:", np.allclose(J, A[:5], atol=1e-6))

scalar function
numeric: [-1.19  0.94 -1.74]
exact  : [-1.19  0.94 -1.74]
match  : True

vector function (5 outputs, 3 inputs)
jacobian shape: (5, 3)
equals first 5 rows of A: True


## Benchmark: vectorised NumPy against Python loops
Each vectorised function is timed against a plain Python loop that does the same
work. `compare` first checks that both give the same answer, then reports the
fastest of several runs. The data uses a fixed seed, so the table can be
reproduced with `python -m numcompute.benchmarking`.

In [26]:
for key, value in benchmarking.environment_info().items():
    print(f"{key}: {value}")

display(Markdown(benchmarking.format_table(benchmarking.run_all(repeat=3))))

Python: 3.15.1
NumPy: 2.5.4
OS: macOS-26.5.1-arm64-arm-64bit-Mach-O
CPU: arm
CPU cores: 15


| Benchmark | Vectorised (ms) | Python loop (ms) | Speedup |
|---|---:|---:|---:|
| stats.mean, (2000, 50), axis=0 | 0.024 | 1.561 | 64x |
| stats.var, (2000, 50), axis=0 | 0.090 | 4.260 | 47x |
| rank.rank (average), 5000 values | 0.177 | 3.203 | 18x |
| sort_search.binary_search, 10000 queries in 1000000 values | 0.223 | 8.572 | 38x |
| metrics.confusion_matrix, 100000 samples, 5 classes | 0.841 | 1.913 | 2x |
| utils.euclidean_distance, (200, 10) vs (200, 10) | 0.472 | 11.477 | 24x |
| utils.softmax, (2000, 10), axis=1 | 0.122 | 1.048 | 9x |